# 57 — Prompt Versioning
**Goal:** Track prompt versions with a registry, golden test sets, and regression checks.

Ch. 56 produced prompt templates; this chapter treats them as **versioned artifacts**. `register_prompt()` stores each prompt's system text, template, test cases, and a content hash in a registry keyed `name@version`, so you can always say *which* prompt produced *which* output — the reproducibility requirement for a system whose answers influence hiring decisions.

**Why it matters for resumes / ATS:** an untracked prompt change can silently degrade extraction quality across every resume processed. With a registry, a "small tweak" becomes a diffable, testable event: register v2.0, run both versions against the golden set, and only promote v2.0 if it wins. Versioning is what lets prompt engineering scale past one person's notebook.

## 1. Prompt Registry

A registry is a dict keyed by `name@version`. Each entry captures the **system prompt**, the **template**, the **test cases** that define correct behavior, and a **hash** of the prompt content — a fingerprint that changes whenever the text changes.

**What the code does:**
- `register_prompt(name, version, system_prompt, template, test_cases)` — builds an entry and stores it under `f"{name}@v{version}"`; the hash is `sha256(system_prompt + template).hexdigest()[:12]`, so two entries with identical content are detectable by eye.
- Registers `skill_extract@v1.0` (with test `("Python developer", ["Python"])`) and `bullet_rewrite@v1.0` (with test `("Responsible for ML", "Developed ML models achieving...")`).

**Expected (verified by running):** the registry ends with 2 entries; the printed keys are `skill_extract@v1.0` and `bullet_rewrite@v1.0` with hashes `150ca689d6c8` and `f4b34b5fada2`. Note the test cases are stored *with* the prompt — a prompt without its tests is a claim without evidence.

In [ ]:
import json, hashlib
PROMPT_REGISTRY = {}

def register_prompt(name, version, system_prompt, template, test_cases=None):
    """Register a prompt with version and tests."""
    entry = {
        "name": name,
        "version": version,
        "system_prompt": system_prompt,
        "template": template,
        "test_cases": test_cases or [],
        "hash": hashlib.sha256((system_prompt + template).encode()).hexdigest()[:12],
    }
    PROMPT_REGISTRY[f"{name}@v{version}"] = entry
    print(f"  Registered: {name}@v{version} (hash: {entry['hash']})")

register_prompt("skill_extract", "1.0", 
    "Extract programming skills from resume text.",
    "Resume: {text}\nSkills:",
    test_cases=[("Python developer", ["Python"])])

register_prompt("bullet_rewrite", "1.0",
    "Rewrite resume bullets in STAR format.",
    "Original: {text}\nRewritten:",
    test_cases=[("Responsible for ML", "Developed ML models achieving...")])

print(f"\nRegistry has {len(PROMPT_REGISTRY)} entries")
for k, v in PROMPT_REGISTRY.items():
    print(f"  {k}: {v['hash']} — {v['system_prompt'][:40]}...")

## 2. Version Comparison

Prompts change, and each change is a **hypothesis**: "this wording improves extraction." The comparison protocol makes that hypothesis testable — run old and new against the same golden set and compare on accuracy, cost, and consistency.

**What the code does:** prints the diffing workflow: register the new version (`skill_extract@v2.0`), run both versions on the golden test set, compare metrics (accuracy, cost, consistency), and only promote v2.0 to default if it is >= v1.0 on *all* metrics. The closing rules are the operational ones: **keep old versions** (rollback is free if you never delete) and **always test before deploying**.

**Why it matters:** a prompt that improves F1 by 2 points but doubles token cost is a business decision, not a code change — and you can only make that call if both versions exist with measured numbers.

In [ ]:
print('''Prompt version diffing:
When updating a prompt:
1. Register new version (e.g., skill_extract@v2.0)
2. Run both old and new against the golden test set
3. Compare outputs: accuracy, cost, consistency
4. If v2.0 >= v1.0 on all metrics, promote to default

Always keep old versions — regression disasters happen.
Always test before deploying new prompt versions.''')

## Summary: Version prompts like code. Registry + golden tests prevent regression.

**Every prompt change is a deploy — give it a version, a hash, and a test run.**

The registry keys prompts as `name@version`, fingerprints content with a SHA-256 hash, and stores test cases alongside the text, so any output can be traced to an exact prompt revision and any change can be validated against the golden set before it becomes the default. That is the difference between prompt *editing* and prompt *engineering*.

This chapter feeds Ch. 58, where the output side gets the same rigor: instead of trusting free-form LLM prose, the response is forced into a JSON contract and validated with Pydantic.